# C12-classical-models — Practice p28 — Solution


The pre-order ledger enumerates every valid candidate at the root and at the only visited nonpure child. The depth limit then turns both grandchildren into deterministic majority leaves.


In [ ]:
import numpy as np

X_p28 = np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.],
                  [2.,0.],[2.,1.],[3.,0.],[3.,1.]], dtype=np.float64)
y_p28 = np.array([0,0,1,0,1,1,0,1], dtype=np.int64)


tree_p28 = {"feature":0,"threshold":0.5,"left":{"prediction":0},
            "right":{"feature":0,"threshold":1.5,
                     "left":{"prediction":0},"right":{"prediction":1}}}
split_ledger_p28 = (
 {"node_path":(),"candidates":(
  {"feature":0,"threshold":0.5,"left_counts":(2,0),"right_counts":(2,4),"weighted_impurity":1/3,"gain":1/6},
  {"feature":0,"threshold":1.5,"left_counts":(3,1),"right_counts":(1,3),"weighted_impurity":3/8,"gain":1/8},
  {"feature":0,"threshold":2.5,"left_counts":(3,3),"right_counts":(1,1),"weighted_impurity":1/2,"gain":0.0},
  {"feature":1,"threshold":0.5,"left_counts":(2,2),"right_counts":(2,2),"weighted_impurity":1/2,"gain":0.0})},
 {"node_path":("right",),"candidates":(
  {"feature":0,"threshold":1.5,"left_counts":(1,1),"right_counts":(1,3),"weighted_impurity":5/12,"gain":1/36},
  {"feature":0,"threshold":2.5,"left_counts":(1,3),"right_counts":(1,1),"weighted_impurity":5/12,"gain":1/36},
  {"feature":1,"threshold":0.5,"left_counts":(1,2),"right_counts":(1,2),"weighted_impurity":4/9,"gain":0.0})})


def predict_certified_tree(tree, X):
    X=np.asarray(X,dtype=np.float64)
    if X.ndim!=2 or not np.isfinite(X).all(): raise ValueError("finite matrix required")
    predictions=np.empty(X.shape[0],dtype=np.int64)
    for index,row in enumerate(X):
        node=tree
        while "prediction" not in node:
            node=node["left"] if row[node["feature"]]<=node["threshold"] else node["right"]
        predictions[index]=int(node["prediction"])
    return predictions


def trace_tree_paths(tree, X):
    X=np.asarray(X,dtype=np.float64)
    traces=[]
    for row in X:
        node=tree; steps=[]
        while "prediction" not in node:
            direction="left" if row[node["feature"]]<=node["threshold"] else "right"
            steps.append((int(node["feature"]),float(node["threshold"]),direction))
            node=node[direction]
        traces.append({"steps":tuple(steps),"prediction":int(node["prediction"])})
    return tuple(traces)


predictions_p28 = predict_certified_tree(tree_p28, X_p28)
paths_p28 = trace_tree_paths(tree_p28, X_p28)
certificates_p28 = {"max_depth":2,"nonempty_children":True,"positive_gains":True,
                    "deterministic_ties":True,"training_predictions":predictions_p28.copy()}


### Answer check


In [ ]:
ATOL=1e-12
RTOL=1e-10
assert set(tree_p28)=={"feature","threshold","left","right"}
assert np.array_equal(predictions_p28,[0,0,0,0,1,1,1,1])
assert paths_p28[0]=={"steps":((0,0.5,"left"),),"prediction":0}
assert paths_p28[2]=={"steps":((0,0.5,"right"),(0,1.5,"left")),"prediction":0}
assert len(split_ledger_p28)==2 and split_ledger_p28[1]["node_path"]==("right",)
assert np.isclose(split_ledger_p28[0]["candidates"][0]["gain"],1/6,atol=ATOL,rtol=RTOL)
assert np.isclose(split_ledger_p28[1]["candidates"][0]["gain"],1/36,atol=ATOL,rtol=RTOL)
assert set(certificates_p28)=={"max_depth","nonempty_children","positive_gains","deterministic_ties","training_predictions"}
